# B1: Amazon Report Data Pipeline

> **Difficulty**: ⭐ Beginner | **Estimated Time**: 30 minutes | **Prerequisites**: Python basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/b1-data-pipeline.ipynb)

This notebook demonstrates how to process Amazon business report data with pandas:
- Reading Excel report files
- Data cleaning and formatting
- Calculating core metrics (GMS, Units, ASP)
- Generating a summary report

---

📎 Source: [ecommerce-ai-roadmap](https://github.com/kangise/ecommerce-ai-roadmap) — Path B1 Data Collection & Processing

In [ ]:
# Install dependencies
!pip install pandas openpyxl

## Background

In cross-border e-commerce operations, Amazon provides various business reports (Business Report, Advertising Report, etc.) typically in Excel or CSV format.

Manual processing of these reports is time-consuming and error-prone. This notebook shows how to automate the workflow with Python + pandas.

### Key Metrics

| Metric | Meaning | Formula |
|--------|---------|----------|
| GMS (Gross Merchandise Sales) | Total sales amount | Unit Price × Units |
| Units | Number of units sold | Direct count |
| ASP (Average Selling Price) | Average selling price | GMS ÷ Units |

## Step 1: Create Sample Data

We'll first create a simulated Amazon Business Report to mimic the real report structure.

In [ ]:
import pandas as pd
import numpy as np

# Create sample data
np.random.seed(42)

n_rows = 50
categories = ['Electronics', 'Home & Kitchen', 'Sports', 'Beauty', 'Toys']
markets = ['US', 'DE', 'JP']

data = {
    'ASIN': [f'B0{np.random.randint(10000000, 99999999)}' for _ in range(n_rows)],
    'Product Name': [f'Product {i+1} - Sample Item' for i in range(n_rows)],
    'Category': np.random.choice(categories, n_rows),
    'Market': np.random.choice(markets, n_rows),
    'Units Ordered': np.random.randint(0, 500, n_rows),
    'Unit Price': np.round(np.random.uniform(9.99, 199.99, n_rows), 2),
    'Sessions': np.random.randint(50, 5000, n_rows),
    'Page Views': np.random.randint(100, 10000, n_rows),
    'Buy Box %': np.round(np.random.uniform(0.5, 1.0, n_rows), 2),
    'Date': pd.date_range('2025-01-01', periods=n_rows, freq='D').strftime('%Y-%m-%d').tolist()
}

# Add some dirty data for cleaning demo
data['Units Ordered'][3] = -5       # Negative number
data['Unit Price'][7] = 0           # Zero price
data['Product Name'][10] = ''       # Empty name
data['Product Name'][15] = None     # None value

df_raw = pd.DataFrame(data)

# Save as Excel
df_raw.to_excel('sample_business_report.xlsx', index=False)
print(f'✅ Sample report created: sample_business_report.xlsx ({len(df_raw)} rows)')
df_raw.head()

## Step 2: Read Excel File

Use pandas + openpyxl to read the Excel report.

In [ ]:
# Read Excel file
df = pd.read_excel('sample_business_report.xlsx', engine='openpyxl')

print(f'📊 Shape: {df.shape}')
print(f'📊 Columns: {list(df.columns)}')
print()
print('--- Data Types ---')
print(df.dtypes)
print()
print('--- Missing Values ---')
print(df.isnull().sum())

## Step 3: Data Cleaning

Common data issues in real reports:
- Missing values / None
- Negative numbers (returns or data errors)
- Zero prices (giveaways or errors)

In [ ]:
df_clean = df.copy()

# 1. Handle missing values
missing_before = df_clean.isnull().sum().sum()
df_clean['Product Name'] = df_clean['Product Name'].fillna('Unknown Product')
df_clean['Product Name'] = df_clean['Product Name'].replace('', 'Unknown Product')

# 2. Filter invalid data
invalid_units = (df_clean['Units Ordered'] < 0).sum()
df_clean = df_clean[df_clean['Units Ordered'] >= 0]

invalid_price = (df_clean['Unit Price'] <= 0).sum()
df_clean = df_clean[df_clean['Unit Price'] > 0]

print(f'🧹 Cleaning Results:')
print(f'   Missing values fixed: {missing_before}')
print(f'   Negative units removed: {invalid_units}')
print(f'   Zero price removed: {invalid_price}')
print(f'   Rows after cleaning: {len(df_clean)} (original: {len(df)})')

## Step 4: Calculate Core Metrics

Calculate GMS for each row, then aggregate by Category and Market.

In [ ]:
# Calculate GMS
df_clean['GMS'] = df_clean['Units Ordered'] * df_clean['Unit Price']

# Calculate conversion rate
df_clean['Conversion Rate'] = np.where(
    df_clean['Sessions'] > 0,
    df_clean['Units Ordered'] / df_clean['Sessions'],
    0
)

print('--- New Metrics Preview ---')
df_clean[['ASIN', 'Units Ordered', 'Unit Price', 'GMS', 'Sessions', 'Conversion Rate']].head(10)

## Step 5: Aggregate by Dimensions

In [ ]:
# Aggregate by Category
category_summary = df_clean.groupby('Category').agg(
    Total_Units=('Units Ordered', 'sum'),
    Total_GMS=('GMS', 'sum'),
    Avg_Price=('Unit Price', 'mean'),
    Total_Sessions=('Sessions', 'sum'),
    ASIN_Count=('ASIN', 'nunique')
).round(2)

# Calculate ASP
category_summary['ASP'] = (category_summary['Total_GMS'] / category_summary['Total_Units']).round(2)

# Sort by GMS
category_summary = category_summary.sort_values('Total_GMS', ascending=False)

print('📊 Summary by Category')
print('=' * 80)
category_summary

In [ ]:
# Aggregate by Market
market_summary = df_clean.groupby('Market').agg(
    Total_Units=('Units Ordered', 'sum'),
    Total_GMS=('GMS', 'sum'),
    Avg_Conversion=('Conversion Rate', 'mean'),
    ASIN_Count=('ASIN', 'nunique')
).round(2)

market_summary['ASP'] = (market_summary['Total_GMS'] / market_summary['Total_Units']).round(2)
market_summary = market_summary.sort_values('Total_GMS', ascending=False)

print('📊 Summary by Market')
print('=' * 80)
market_summary

## Step 6: Generate Summary Report

In [ ]:
# Overall summary
total_gms = df_clean['GMS'].sum()
total_units = df_clean['Units Ordered'].sum()
overall_asp = total_gms / total_units if total_units > 0 else 0
avg_conversion = df_clean['Conversion Rate'].mean()

print('=' * 60)
print('📋 Business Report Summary')
print('=' * 60)
print(f'  Total GMS:              ${total_gms:,.2f}')
print(f'  Total Units:            {total_units:,}')
print(f'  ASP:                    ${overall_asp:.2f}')
print(f'  Avg Conversion Rate:    {avg_conversion:.2%}')
print(f'  Unique ASINs:           {df_clean["ASIN"].nunique()}')
print(f'  Markets:                {", ".join(df_clean["Market"].unique())}')
print(f'  Clean Rows:             {len(df_clean)}')
print('=' * 60)

# Export cleaned data
df_clean.to_excel('cleaned_report.xlsx', index=False)
print(f'\n✅ Cleaned data exported: cleaned_report.xlsx')

## Summary

This notebook demonstrated the basic workflow for processing Amazon report data:

1. **Read data**: Use `pd.read_excel()` with the openpyxl engine to read Excel files
2. **Data cleaning**: Handle missing values, filter invalid data
3. **Metric calculation**: GMS = Units × Price, ASP = GMS ÷ Units, CVR = Units ÷ Sessions
4. **Multi-dimensional aggregation**: Aggregate by Category, Market, and other dimensions
5. **Export report**: Output the cleaned Excel file

### Next Steps

- 📈 [B2: Prediction Models & Decision Making](../paths/b-developers/b2-prediction-models.md) — Sales forecasting with Prophet
- 📖 [Full Path B](../paths/b-developers/README.md) — View the developer learning path
- 🏠 [Back to Hub](https://github.com/kangise/ecommerce-ai-roadmap)

---

📎 **Source**: [ecommerce-ai-roadmap](https://github.com/kangise/ecommerce-ai-roadmap) — AI × Cross-Border E-Commerce Knowledge Base